# <a href="https://girafe.ai/" target="_blank" rel="noopener noreferrer"><img src="https://raw.githubusercontent.com/girafe-ai/ml-course/7096a5df4cada5ee651be1e3215c2f7fb8a7e0bf/logo_margin.svg" alt="girafe-ai logo" width="150px" align="left"></a> YMLS Uzbekistan, ML Fundamentals, fall 2026 <a class="tocSkip">

# Week 6 seminar. Naive Bayes and probabilistic classifiers

The setting is a listing site where some of the listings are fakes. Bayes' rule on one word comes first, then naive Bayes by hand on eight listings. Multinomial naive Bayes on listing texts and Gaussian naive Bayes on three classes of housing are written from scratch and checked against scikit-learn. The last two sections measure what the independence assumption costs and compare kNN, logistic regression and naive Bayes on one dataset.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/girafe-ai/ml-course/blob/26f_ymls_uz/week0_06_naive_bayes/week0_06_01_naive_bayes.ipynb)

This week's [lecture slides](https://github.com/girafe-ai/ml-course/blob/26f_ymls_uz/week0_06_naive_bayes/ml-basic_lect06_naive_bayes.pdf) are in the same folder. The links to the interactive pages are in the course chat.

_YMLS Uzbekistan, ML Fundamentals, fall 2026_

Fill in the cells marked `# YOUR CODE HERE`. The cell after each exercise checks your answer and says what is wrong. Run the cells from top to bottom, since later sections use earlier variables. It takes about 80 minutes.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


SEED = 42  # one seed for every split


def sigmoid(z):
    """1 / (1 + exp(-z)), written so that a large |z| does not overflow."""
    return np.exp(-np.logaddexp(0, -z))


def log_loss(y, p):
    """Mean negative log-likelihood of the labels y (0 or 1) under the probabilities p."""
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), 1e-15, 1 - 1e-15)  # log(0) is minus infinity
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


def confusion(y, flagged):
    """TP, FP, FN, TN for the labels y (1 is the positive class) and the flags of a model."""
    y, flagged = np.asarray(y).astype(bool), np.asarray(flagged).astype(bool)
    tp, fp = int((flagged & y).sum()), int((flagged & ~y).sum())
    fn, tn = int((~flagged & y).sum()), int((~flagged & ~y).sum())
    return tp, fp, fn, tn


# Plot style: gray axes, colour only for data
INK, INK2, INK3, LINE = "#1D1D1F", "#6E6E73", "#86868B", "#D2D2D7"
ACCENT, ORANGE, GREEN, PURPLE = "#0071E3", "#FF9F0A", "#34C759", "#AF52DE"
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica Neue", "Arial", "DejaVu Sans"],
        "font.size": 13,
        "mathtext.fontset": "cm",
        "axes.edgecolor": LINE,
        "axes.linewidth": 0.8,
        "axes.labelcolor": INK2,
        "axes.titlecolor": INK,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "axes.prop_cycle": plt.cycler(color=[ACCENT, ORANGE, GREEN, PURPLE, INK3]),
        "xtick.color": INK3,
        "ytick.color": INK3,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "xtick.minor.size": 0,
        "ytick.minor.size": 0,
        "legend.frameon": False,
        "legend.fontsize": 11,
        "lines.linewidth": 2.2,
        "lines.markersize": 6,
        "figure.facecolor": "white",
    }
)

Three helpers from earlier weeks are defined above: the sigmoid and the log-loss of week 4 and the four counts of the confusion matrix of week 5.

## 1. Bayes' rule in numbers

On a listing site 5% of the listings are fakes: a flat that does not exist, posted to collect a prepayment. The word "prepayment" appears in 40% of the fake listings and in 2% of the real ones. A new listing contains the word. How likely is it to be a fake?

Bayes' rule turns the question around:

$$P(\text{fake} \mid \text{word}) = \frac{P(\text{fake}) \, P(\text{word} \mid \text{fake})}{P(\text{word})}, \qquad P(\text{word}) = P(\text{fake}) \, P(\text{word} \mid \text{fake}) + P(\text{real}) \, P(\text{word} \mid \text{real}).$$

$P(\text{fake})$ is the prior, $P(\text{word} \mid \text{fake})$ is the likelihood, $P(\text{word})$ is the evidence and $P(\text{fake} \mid \text{word})$ is the posterior.

**Exercise 1.1.** Write `posterior(prior, like_fake, like_real)`: the probability that a listing is a fake, given that it contains the word.

In [ ]:
def posterior(prior, like_fake, like_real):
    """P(fake | word) from the prior P(fake) and the likelihoods P(word | fake), P(word | real)."""
    # YOUR CODE HERE: the evidence P(word) is prior * like_fake + (1 - prior) * like_real;
    # return prior * like_fake divided by the evidence
    return None

In [ ]:
answer = posterior(0.05, 0.40, 0.02)
assert answer is not None, "Fill in posterior above"
assert np.isclose(
    answer, 20 / 39
), "Expect 20 / 39 = 0.513: prior times likelihood, over the evidence"

fakes, real = 0.05 * 1000, 0.95 * 1000
print(f"in 1,000 listings: {fakes:.0f} fakes, {0.40 * fakes:.0f} of them with the word")
print(f"                   {real:.0f} real listings, {0.02 * real:.0f} of them with the word")
print(f"P(fake | prepayment) = {answer:.3f}")

Among 1,000 listings there are 50 fakes, and 20 of them contain the word. Of the 950 real listings, 19 contain it. So 20 of the 39 listings with the word are fakes, which is 0.513. The word is 20 times more likely in a fake listing and the answer is still close to a coin flip, because real listings outnumber fakes 19 to 1.

Now take the same word to sites with other shares of fakes.

In [ ]:
assert posterior(0.05, 0.40, 0.02) is not None, "Finish Exercise 1.1 first"
print("share of fakes   P(fake | prepayment)")
for prior in [0.001, 0.01, 0.05, 0.2, 0.5]:
    print(f"{prior:14.3f} {posterior(prior, 0.40, 0.02):22.3f}")

prior_grid = np.logspace(-3, 0, 200)  # 0.001 ... 1
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(prior_grid, prior_grid, color=INK3, ls="--", lw=1.2, label="a word that says nothing")
ax.plot(prior_grid, posterior(prior_grid, 0.40, 0.02), color=ACCENT, label='"prepayment"')
ax.scatter([0.05], [posterior(0.05, 0.40, 0.02)], s=70, color=ACCENT, edgecolor="white", zorder=4)
ax.annotate(
    "5% fakes: 0.513",
    (0.05, 0.513),
    xytext=(-12, 6),
    textcoords="offset points",
    ha="right",
    fontsize=11,
    color=INK2,
)
ax.set(
    xscale="log",
    xlabel="share of fakes on the site (the prior)",
    ylabel="P(fake | the word)",
    title="One word, different priors",
)
ax.legend(loc="upper left")
plt.show()

With a prior of 0.5 the same word gives 0.952, and with one fake in a thousand it gives 0.020. The likelihoods are the same in every row, only the prior moved. Week 5 had the same effect under another name: the precision of a classifier is the posterior $P(\text{fake} \mid \text{flagged})$, and it fell when fakes became rarer. The page `01_bayes_rule.html` has the three numbers as sliders.

## 2. Naive Bayes by hand

A listing has many words. Call the whole listing $x$ and the classes $c$. The Bayes classifier picks the class with the largest $P(c) \, P(x \mid c)$, and the hard part is $P(x \mid c)$. For 10 yes/no features the full table has $2^{10} - 1 = 1023$ numbers per class. Naive Bayes assumes that the features are independent given the class,

$$P(x \mid c) = \prod_j P(x_j \mid c),$$

which leaves 10 numbers per class.

Eight listings have been labelled by hand: 3 fakes and 5 real ones. The table below counts how many times each of four words occurs in the listings of each class.

In [ ]:
WORDS = ["urgent", "prepayment", "metro", "balcony"]
word_counts = np.array([[2, 0, 10, 8], [5, 4, 1, 2]])  # row 0: real listings, row 1: fake listings
listings = np.array([5, 3])  # listings per class: real, fake
priors = listings / listings.sum()  # P(real), P(fake)

print(f"{'word':12s} {'in fake listings':>17s} {'in real listings':>17s}")
for j, word in enumerate(WORDS):
    print(f"{word:12s} {word_counts[1, j]:17d} {word_counts[0, j]:17d}")
print(f"{'all words':12s} {word_counts[1].sum():17d} {word_counts[0].sum():17d}")
print(f"\npriors: fake {priors[1]:.3f}, real {priors[0]:.3f}")

Everywhere in this notebook class 0 is "real" and class 1 is "fake", so row 0 of a table belongs to the real listings.

For texts the model is multinomial naive Bayes. Let $N_{cj}$ be the number of times word $j$ occurs in the texts of class $c$, $N_c = \sum_j N_{cj}$ the number of words in the class, and $V$ the size of the vocabulary. The probability of word $j$ in class $c$ is estimated as

$$\theta_{cj} = \frac{N_{cj} + \alpha}{N_c + \alpha V}.$$

With $\alpha = 0$ this is the maximum likelihood estimate of week 2, a relative frequency. With $\alpha = 1$ every word gets one extra count in every class, which is called Laplace smoothing.

**Exercise 2.1.** Write `word_probabilities(counts, alpha)`: the matrix of $\theta_{cj}$ with one row per class.

In [ ]:
def word_probabilities(counts, alpha):
    """theta[c, j] = (N[c, j] + alpha) / (N[c] + alpha * V) for a matrix of counts N[c, j]."""
    # YOUR CODE HERE: counts.sum(axis=1, keepdims=True) is N[c] as a column,
    # V is the number of columns, counts.shape[1]
    return None

In [ ]:
theta_ml = word_probabilities(word_counts, 0)  # alpha = 0: maximum likelihood
assert theta_ml is not None, "Fill in word_probabilities above"
theta_smooth = word_probabilities(word_counts, 1)  # alpha = 1: Laplace smoothing
assert theta_ml.shape == (2, 4), "Expect one row per class and one column per word: (2, 4)"
assert np.allclose(
    theta_ml[1], [5 / 12, 4 / 12, 1 / 12, 2 / 12]
), "With alpha = 0 the fake row is 5/12, 4/12, 1/12, 2/12: every count over the row total"
assert np.allclose(theta_smooth[0], [3 / 24, 1 / 24, 11 / 24, 9 / 24]), (
    "With alpha = 1 the real row is 3/24, 1/24, 11/24, 9/24: add alpha to every count "
    "and alpha * V to the row total"
)
for alpha, theta in [(0, theta_ml), (1, theta_smooth)]:
    print(f"alpha = {alpha}      " + " ".join(f"{word:>11s}" for word in WORDS))
    print("  theta, fake" + " ".join(f"{value:11.3f}" for value in theta[1]))
    print("  theta, real" + " ".join(f"{value:11.3f}" for value in theta[0]))

Without smoothing, "prepayment" has a probability of exactly 0 in the real class, because the word never occurred in a real listing.

A new listing arrives: "urgent prepayment metro". As a vector of word counts it is $x = (1, 1, 1, 0)$. A text with the counts $x_j$ has $P(x \mid c) \propto \prod_j \theta_{cj}^{x_j}$, so the score of a class is

$$\text{score}(c) = P(c) \prod_j \theta_{cj}^{x_j},$$

and the posterior of a class is its score divided by the sum of the scores.

**Exercise 2.2.** Write `class_scores(x, theta, priors)`: the scores of the two classes as an array.

In [ ]:
def class_scores(x, theta, priors):
    """P(c) times the product over the words of theta[c, j] ** x[j], for every class c."""
    # YOUR CODE HERE: theta**x raises every column to the power of its count,
    # np.prod(..., axis=1) multiplies along a row
    return None

In [ ]:
assert word_probabilities(word_counts, 0) is not None, "Finish Exercise 2.1 first"
x_new = np.array([1, 1, 1, 0])  # "urgent prepayment metro" as counts of WORDS
scores_ml = class_scores(x_new, theta_ml, priors)
assert scores_ml is not None, "Fill in class_scores above"
scores_smooth = class_scores(x_new, theta_smooth, priors)
assert np.allclose(
    scores_ml, [0, 0.375 * 5 / 12 * 4 / 12 * 1 / 12]
), "With alpha = 0 expect 0 for real and 0.00434 for fake: prior times the three thetas"
assert np.allclose(
    scores_smooth, [0.001492, 0.005493], atol=5e-7
), "With alpha = 1 expect 0.001492 for real and 0.005493 for fake"
for alpha, scores in [(0, scores_ml), (1, scores_smooth)]:
    print(
        f"alpha = {alpha}: score of fake {scores[1]:.6f}, score of real {scores[0]:.6f}, "
        f"P(fake | listing) = {scores[1] / scores.sum():.3f}"
    )

At $\alpha = 0$ the real class gets a score of 0 and the posterior of "fake" is 1. One unseen word vetoes the class, whatever the other words say: "metro" is six times more likely in a real listing, and it changes nothing. At $\alpha = 1$ the scores are 0.005493 and 0.001492, and the posterior of "fake" is 0.786. Smoothing is a MAP estimate with a Dirichlet prior on $\theta$. Week 3 had the same idea for regression, where a prior on the weights became a penalty. The page `02_naive_bayes_by_hand.html` lets you build other listings and move $\alpha$.

### 2.1 The same in logarithms

A product of many small numbers underflows, as the likelihood did in week 2. Take the logarithm and the product becomes a sum:

$$\log \text{score}(c) = \log P(c) + \sum_j x_j \log \theta_{cj}.$$

With two classes the posterior comes back through the sigmoid: $P(\text{fake} \mid x) = \sigma \big( \log \text{score}(\text{fake}) - \log \text{score}(\text{real}) \big)$.

**Exercise 2.3.** Write `class_log_scores(x, theta, priors)`.

In [ ]:
def class_log_scores(x, theta, priors):
    """log P(c) plus the sum over the words of x[j] * log theta[c, j], for every class c."""
    # YOUR CODE HERE: np.log(theta).T has one column per class, so x @ np.log(theta).T
    # is the sum over the words for both classes at once; add np.log(priors)
    return None

In [ ]:
assert word_probabilities(word_counts, 0) is not None, "Finish Exercise 2.1 first"
assert (
    class_scores(np.zeros(4), np.full((2, 4), 0.25), priors) is not None
), "Finish Exercise 2.2 first"
log_scores = class_log_scores(x_new, theta_smooth, priors)
assert log_scores is not None, "Fill in class_log_scores above"
assert np.allclose(
    np.exp(log_scores), scores_smooth
), "exp of the log-scores must give the scores of Exercise 2.2"
print(f"log-scores: fake {log_scores[1]:.3f}, real {log_scores[0]:.3f}")
print(f"P(fake | listing) = {sigmoid(log_scores[1] - log_scores[0]):.3f}")

x_long = 300 * x_new  # the same three words, 300 times each
long_log_scores = class_log_scores(x_long, theta_smooth, priors)
print("\nthe listing repeated 300 times")
print("  products (real, fake):", class_scores(x_long, theta_smooth, priors))
print(f"  log-scores: fake {long_log_scores[1]:.1f}, real {long_log_scores[0]:.1f}")

# scikit-learn needs only the counts per class, so one row per class is enough to fit it
sk_hand = MultinomialNB(alpha=1.0, class_prior=priors).fit(word_counts, [0, 1])
print(f"\nMultinomialNB: P(fake | listing) = {sk_hand.predict_proba([x_new])[0, 1]:.3f}")

The log-scores are $-5.204$ for "fake" and $-6.508$ for "real", and they give the same 0.786. For the listing repeated 300 times both products are 0.0 in floating point and there is nothing left to compare, while the log-scores, $-1268.0$ and $-1811.8$, still say "fake". `MultinomialNB` from scikit-learn works in logarithms too and returns the same posterior.

## 3. Listing texts: multinomial naive Bayes

Now a larger site: 200 labelled listings to train on and 1,000 new ones to test on. A quarter of them are fakes. The vocabulary has ten words, and a listing is a bag of words: the model sees how many times each word occurs and ignores the order. The data is synthetic. Every listing has 4 + Poisson(6) words, each drawn from the word distribution of its class.

In [ ]:
VOCAB = [
    "urgent",
    "prepayment",
    "cheap",
    "call",
    "deposit",
    "owner",
    "metro",
    "photos",
    "renovated",
    "balcony",
]
WORD_P = {  # the probability of each word of VOCAB in a listing of the class
    "fake": [0.20, 0.18, 0.18, 0.14, 0.10, 0.06, 0.06, 0.03, 0.03, 0.02],
    "real": [0.03, 0.01, 0.08, 0.10, 0.06, 0.12, 0.18, 0.12, 0.16, 0.14],
}


def text_sample(seed, n, share=0.25):
    """n listings as word counts over VOCAB: X has shape (n, 10), y is 1 (fake) or 0 (real)."""
    rng = np.random.default_rng(seed)
    y = np.zeros(n, dtype=int)
    y[: int(round(share * n))] = 1  # exactly this share of fakes
    rng.shuffle(y)
    length = 4 + rng.poisson(6, n)  # the number of words in each listing
    X = np.array(
        [
            rng.multinomial(m, WORD_P["fake" if c else "real"])
            for m, c in zip(length, y, strict=True)
        ]
    )
    return X, y


def counts_to_text(row):
    """A row of counts as a string, to look at: 'urgent urgent cheap call'."""
    return " ".join(word for word, k in zip(VOCAB, row, strict=True) for _ in range(int(k)))


X_text, y_text = text_sample(3, 200)  # the 200 listings we train on
X_text_test, y_text_test = text_sample(5003, 1000)  # 1,000 new listings

print(
    f"train: {X_text.shape[0]} listings, {y_text.sum()} fakes; "
    f"test: {X_text_test.shape[0]} listings, {y_text_test.sum()} fakes"
)
for label, name in [(1, "fake"), (0, "real")]:
    print(f"\n{name}:")
    for row in X_text[y_text == label][:3]:
        print("  ", counts_to_text(row))
print("\nthe first listing as a row of counts:", X_text[0])

Training is counting, as in section 2.

**Exercise 3.1.** Fill in `N_text`, a 2 by 10 matrix whose row $c$ is the sum of the count vectors of the training listings of class $c$, and `prior_text`, the share of each class among the training listings.

In [ ]:
# YOUR CODE HERE: X_text[y_text == c] holds the listings of class c; sum them over axis 0
# for c = 0 and c = 1. The prior of class c is np.mean(y_text == c)
N_text = None
prior_text = None

In [ ]:
assert N_text is not None and prior_text is not None, "Fill in N_text and prior_text above"
assert word_probabilities(word_counts, 0) is not None, "Finish Exercise 2.1 first"
N_text, prior_text = np.asarray(N_text), np.asarray(prior_text)
assert N_text.shape == (2, 10), "Expect one row per class and one column per word: (2, 10)"
assert N_text[1].tolist() == [
    93,
    104,
    95,
    61,
    52,
    35,
    27,
    15,
    16,
    9,
], "Row 1 is the sum of the count vectors of the fake listings: X_text[y_text == 1].sum(axis=0)"
assert N_text[0].sum() == 1463, "Row 0 is the sum over the real listings: 1,463 words in total"
assert np.allclose(prior_text, [0.75, 0.25]), "Expect the priors 0.75 (real) and 0.25 (fake)"

theta_text = word_probabilities(N_text, alpha=1)  # the function of Exercise 2.1
print(f"{'word':12s} {'N, fake':>8s} {'N, real':>8s} {'theta, fake':>12s} {'theta, real':>12s}")
for j, word in enumerate(VOCAB):
    print(
        f"{word:12s} {N_text[1, j]:8d} {N_text[0, j]:8d} "
        f"{theta_text[1, j]:12.4f} {theta_text[0, j]:12.4f}"
    )
print(f"{'all words':12s} {N_text[1].sum():8d} {N_text[0].sum():8d}")

**Exercise 3.2.** Classify the 1,000 test listings. `class_log_scores` from Exercise 2.3 also accepts a whole matrix of counts and returns one row of two log-scores per listing. Store that matrix in `log_scores_test`, and in `pred_text` the class with the larger log-score for every listing.

In [ ]:
# YOUR CODE HERE: call class_log_scores with X_text_test, theta_text and prior_text,
# then take argmax(axis=1) of the result
log_scores_test = None
pred_text = None

In [ ]:
assert (
    log_scores_test is not None and pred_text is not None
), "Fill in log_scores_test and pred_text above"
assert np.shape(log_scores_test) == (
    1000,
    2,
), "Expect one row per test listing and one column per class: (1000, 2)"
nb_text = MultinomialNB(alpha=1.0).fit(X_text, y_text)  # the same model from scikit-learn
assert np.allclose(
    log_scores_test, nb_text.predict_joint_log_proba(X_text_test)
), "The log-scores differ from MultinomialNB: log prior plus counts times log theta"
assert np.array_equal(
    pred_text, nb_text.predict(X_text_test)
), "pred_text is the number of the column with the larger log-score: argmax(axis=1)"
p_nb_text = sigmoid(log_scores_test[:, 1] - log_scores_test[:, 0])  # P(fake | listing)
tp, fp, fn, tn = confusion(y_text_test, pred_text)
print(
    f"accuracy {np.mean(pred_text == y_text_test):.3f} "
    f"(flag nothing: {np.mean(y_text_test == 0):.3f})"
)
print(f"fakes caught {tp}, false alarms {fp}, fakes missed {fn}, real listings passed {tn}")
print(
    f"precision {tp / (tp + fp):.3f}, recall {tp / (tp + fn):.3f}, "
    f"log-loss {log_loss(y_text_test, p_nb_text):.3f}"
)

print("\nalpha     TP   FP   FN   TN   accuracy")
for alpha in [0.01, 1, 10]:
    flags = MultinomialNB(alpha=alpha).fit(X_text, y_text).predict(X_text_test)
    tp, fp, fn, tn = confusion(y_text_test, flags)
    print(f"{alpha:5g} {tp:6d} {fp:4d} {fn:4d} {tn:4d} {np.mean(flags == y_text_test):10.3f}")

Our classifier and `MultinomialNB(alpha=1)` agree on every log-score. On the test listings it gets 0.983 right, where "flag nothing" gets 0.750. It catches 242 fakes, misses 8 and raises 9 false alarms, so the precision is 0.964 and the recall is 0.968. The choice of $\alpha$ matters little here: $\alpha = 0.01$ gives the same four counts, and $\alpha = 10$ misses one more fake and raises one more false alarm. Every word occurs in both classes of the training texts, so there is no zero to repair.

### 3.1 Naive Bayes is a linear classifier

For two classes, subtract the two log-scores:

$$\log \frac{P(\text{fake} \mid x)}{P(\text{real} \mid x)} = \underbrace{\log \frac{P(\text{fake})}{P(\text{real})}}_{b} + \sum_j x_j \underbrace{\log \frac{\theta_{\text{fake}, j}}{\theta_{\text{real}, j}}}_{w_j}.$$

The log-odds are linear in the word counts, as in logistic regression. Here the weights come from counting, and nothing is optimised.

**Exercise 3.3.** Compute the intercept `b_nb` and the vector of weights `w_nb` from `prior_text` and `theta_text`.

In [ ]:
# YOUR CODE HERE: index 1 is "fake" and index 0 is "real" in prior_text and in the rows of
# theta_text; b_nb is one number and w_nb has one weight per word
b_nb = None
w_nb = None

In [ ]:
assert b_nb is not None and w_nb is not None, "Fill in b_nb and w_nb above"
assert log_scores_test is not None, "Finish Exercise 3.2 first"
assert np.isclose(b_nb, np.log(0.25 / 0.75)), "The intercept is log(P(fake) / P(real)) = -1.099"
assert np.shape(w_nb) == (10,), "Expect one weight per word of VOCAB"
log_odds = b_nb + X_text_test @ w_nb  # the linear form
assert np.allclose(
    log_odds, log_scores_test[:, 1] - log_scores_test[:, 0]
), "b + X @ w must equal the difference of the two log-scores: w = log(theta_fake / theta_real)"
assert np.array_equal(log_odds > 0, pred_text == 1), "A positive log-odds means the class fake"

print(f"intercept {b_nb:+.3f}")
for word, weight in zip(VOCAB, w_nb, strict=True):
    print(f"{word:12s} {weight:+.2f}")

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(VOCAB, w_nb, color=[ACCENT if weight > 0 else INK3 for weight in w_nb])
ax.bar_label(bars, fmt="%+.2f", padding=4, fontsize=11, color=INK2)
ax.axvline(0, color=LINE, lw=0.8)
ax.invert_yaxis()
ax.set(
    xlabel="weight, log(theta fake / theta real)",
    xlim=(-2.8, 4.1),
    title="Naive Bayes weights: towards fake (blue), towards real (gray)",
)
plt.show()

The intercept is $\log(0.25 / 0.75) = -1.099$: before reading a word, a listing is three times more likely to be real. "prepayment" has the largest weight, $+3.30$, and "urgent" adds $+1.98$. "balcony" ($-2.03$) and "renovated" ($-1.70$) pull towards "real". "call" is close to zero ($+0.15$) because it is about as common in both classes. The page `03_text_classifier.html` adds up these weights for a listing that you compose.

### 3.2 Logistic regression on the same counts

Logistic regression fits the same linear form by maximising the likelihood of the labels. `C=np.inf` switches its penalty off, as in week 4.

**Exercise 3.4.** Fit `LogisticRegression(C=np.inf, tol=1e-10, max_iter=1_000_000)` on the training counts as `lr_text`, and store its probabilities of "fake" for the test listings in `p_lr_text`.

In [ ]:
# YOUR CODE HERE: fit the model on X_text, y_text; the probabilities of class 1 are
# the second column of predict_proba(X_text_test)
lr_text = None
p_lr_text = None

In [ ]:
assert lr_text is not None and p_lr_text is not None, "Fill in lr_text and p_lr_text above"
nb_text = MultinomialNB(alpha=1.0).fit(X_text, y_text)
p_nb_text = nb_text.predict_proba(X_text_test)[:, 1]
assert (
    abs(np.mean((p_lr_text >= 0.5) == y_text_test) - 0.976) < 0.006
), "Expect an accuracy of about 0.976: fit on X_text, y_text, predict_proba(X_text_test)[:, 1]"
assert log_loss(y_text_test, p_lr_text) > 3 * log_loss(
    y_text_test, p_nb_text
), "Expect a log-loss several times that of naive Bayes: is the penalty off, C=np.inf?"
lr_default = LogisticRegression(tol=1e-10, max_iter=1_000_000).fit(X_text, y_text)  # C = 1
rows = [
    (
        "naive Bayes, alpha = 1",
        nb_text,
        nb_text.feature_log_prob_[1] - nb_text.feature_log_prob_[0],
    ),
    ("logistic regression, no penalty", lr_text, lr_text.coef_[0]),
    ("logistic regression, C = 1", lr_default, lr_default.coef_[0]),
]
print(
    f"{'model':32s} {'train accuracy':>15s} {'test accuracy':>14s} {'test log-loss':>14s} "
    f"{'largest |weight|':>17s}"
)
for name, model, weights in rows:
    p = model.predict_proba(X_text_test)[:, 1]
    print(
        f"{name:32s} {model.score(X_text, y_text):15.3f} {np.mean((p >= 0.5) == y_text_test):14.3f} "
        f"{log_loss(y_text_test, p):14.3f} {np.abs(weights).max():17.2f}"
    )

wrong = (p_lr_text >= 0.5) != y_text_test
certain = (p_lr_text < 0.001) | (p_lr_text > 0.999)
print(
    f"\nlogistic regression without a penalty: {wrong.sum()} mistakes on the test listings, "
    f"{np.sum(wrong & certain)} of them with a probability below 0.001 or above 0.999"
)

Without a penalty logistic regression gets 0.976 of the test listings right, a little below naive Bayes, and its log-loss is 0.473 against 0.041. It labels all 200 training texts correctly: they are separable, so the weights run away as in week 4, and the largest reaches 50.11. On new listings such a model is certain and sometimes wrong: 19 of its 24 mistakes come with a probability below 0.001 or above 0.999. With the default penalty, $C = 1$, the largest weight is 1.95, the accuracy is 0.980 and the log-loss is 0.054. The weights of naive Bayes are logarithms of ratios of counts, so they cannot run away, and 200 texts are enough to count ten words.

### 3.3 Real strings

Real listings are strings, and something has to turn them into counts. `CountVectorizer` builds the vocabulary from the training texts (lower case, split on spaces and punctuation, one-letter words dropped) and returns the matrix of counts. Here are ten short listings written for this notebook and four new ones.

In [ ]:
ads = [
    "Urgent sale, prepayment today, very cheap",
    "Cheap flat, urgent, send a deposit before viewing",
    "Prepayment only, the owner is abroad, urgent",
    "Cheap and urgent, call now, deposit by card",
    "Two rooms near the metro, renovated, balcony",
    "Renovated flat with a balcony, photos inside",
    "Quiet street, metro in five minutes, call the owner",
    "Bright flat, new photos, balcony and a view of the park",
    "Renovated kitchen, near the metro, call in the evening",
    "Family flat from the owner, balcony, school nearby",
]
ad_labels = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])  # the first four are fakes
new_ads = [
    "Urgent, cheap, prepayment by card",
    "Renovated flat near the metro with a balcony",
    "Cheap flat near the metro, call the owner",
    "Penthouse, swimming pool",
]

**Exercise 3.5.** Create a `CountVectorizer` as `vectoriser` and store the counts of `ads` in `ad_counts` (`fit_transform`). Fit `MultinomialNB()` on them as `ad_model`; its default is $\alpha = 1$. Then turn `new_ads` into counts with the same vectoriser (`transform`) and store the probabilities of "fake" in `p_new_ads`.

In [ ]:
# YOUR CODE HERE: vectoriser.fit_transform(ads) learns the vocabulary and returns the counts;
# vectoriser.transform(new_ads) reuses that vocabulary; predict_proba(...)[:, 1] is P(fake)
vectoriser = None
ad_counts = None
ad_model = None
p_new_ads = None

In [ ]:
assert (
    ad_counts is not None and p_new_ads is not None
), "Fill in vectoriser, ad_counts, ad_model and p_new_ads above"
assert isinstance(vectoriser, CountVectorizer), "vectoriser is a CountVectorizer()"
assert ad_counts.shape == (10, 46), "Expect 10 listings and a vocabulary of 46 words"
assert np.allclose(p_new_ads, [0.998, 0.001, 0.029, 0.400], atol=0.002), (
    "Expect about 0.998, 0.001, 0.029 and 0.400: fit on ad_counts, transform new_ads with "
    "the same vectoriser"
)
vocabulary = vectoriser.get_feature_names_out()
first = ad_counts[0].toarray()[0]  # the sparse row of the first listing as a plain array
print(
    f"{ad_counts.shape[1]} words in the vocabulary, {ad_counts.nnz} of the "
    f"{ad_counts.shape[0] * ad_counts.shape[1]} entries of the matrix are not zero"
)
print("the first listing:", ads[0])
print(
    "its counts:       ",
    ", ".join(f"{word} {k}" for word, k in zip(vocabulary, first, strict=True) if k),
)

known = np.asarray(vectoriser.transform(new_ads).sum(axis=1)).ravel()
print("\nP(fake)   known words   listing")
for text, p, n_known in zip(new_ads, p_new_ads, known, strict=True):
    print(f"{p:7.3f} {n_known:13d}   {text}")

The vocabulary has 46 words. The matrix is sparse: 76 of its 460 entries are not zero, and scikit-learn stores only those. The first two new listings are easy. The third mixes "cheap" with "metro", "call" and "owner" and comes out as real, with 0.029. The fourth has no word from the vocabulary. `transform` ignores words it has not seen, so the row of counts is empty and the answer is the prior, 4 fakes out of 10.

## 4. Numbers: Gaussian naive Bayes

Counts suit words. For a numeric feature, Gaussian naive Bayes puts one normal distribution on every class and feature:

$$P(x_j \mid c) = \frac{1}{\sqrt{2 \pi \sigma_{cj}^2}} \exp \left( -\frac{(x_j - \mu_{cj})^2}{2 \sigma_{cj}^2} \right),$$

where $\mu_{cj}$ and $\sigma_{cj}^2$ are the mean and the variance of feature $j$ among the training objects of class $c$.

The data is the three classes of housing from week 4: economy, comfort and business, described by the area in m² and the price per m² in US dollars. There are 50 flats of each class to train on and 200 of each to test on.

In [ ]:
CLASS_NAMES = ["economy", "comfort", "business"]
CLASS_COLOURS = [ACCENT, ORANGE, GREEN]


def classes_sample(seed, n_per):
    """n_per flats of each class: X = (area in m², price per m² in US dollars), y = 0, 1 or 2."""
    rng = np.random.default_rng(seed)
    blocks = []
    for k, (mean_area, mean_ppm) in enumerate([(45, 900), (65, 1300), (90, 1750)]):
        area = rng.normal(mean_area, 9, n_per)
        # inside a class a larger flat costs more per m²
        ppm = mean_ppm + 12 * (area - mean_area) + rng.normal(0, 130, n_per)
        blocks.append(np.column_stack([np.round(area), np.round(ppm, -1), np.full(n_per, k)]))
    data = np.vstack(blocks)
    return data[:, :2], data[:, 2].astype(int)


X_cls, y_cls = classes_sample(6, 50)  # 150 flats to train on
X_cls_test, y_cls_test = classes_sample(5006, 200)  # 600 new flats

class_corr = [np.corrcoef(X_cls[y_cls == k].T)[0, 1] for k in range(3)]
for name, value in zip(CLASS_NAMES, class_corr, strict=True):
    print(f"{name:9s} correlation of area and price per m² inside the class: {value:.2f}")

fig, ax = plt.subplots(figsize=(8, 4.8))
for k, name in enumerate(CLASS_NAMES):
    ax.scatter(
        *X_cls[y_cls == k].T,
        s=26,
        color=CLASS_COLOURS[k],
        edgecolor="white",
        linewidth=0.5,
        label=name,
    )
ax.set(xlabel="area, m²", ylabel="price per m², $", title="150 flats, three classes")
ax.legend(loc="upper left")
plt.show()

Inside a class a larger flat costs more per m²: the correlation of the two features is 0.67, 0.73 and 0.68. The naive assumption says it is zero.

**Exercise 4.1.** Write `gaussian_nb_fit(X, y)`. It returns three arrays with one row per class: the priors (the share of each class), the means of the features and their variances (`var(axis=0)`, the maximum likelihood estimate).

In [ ]:
def gaussian_nb_fit(X, y):
    """Priors (K,), means (K, d) and variances (K, d) of the features inside each class."""
    # YOUR CODE HERE: for every class c in np.unique(y) take the rows X[y == c];
    # their share is the prior, mean(axis=0) the means, var(axis=0) the variances;
    # return priors, means, variances as three arrays
    return None

In [ ]:
fitted = gaussian_nb_fit(X_cls, y_cls)
assert fitted is not None, "Fill in gaussian_nb_fit above"
cls_priors, cls_means, cls_vars = (np.asarray(part) for part in fitted)
assert cls_means.shape == (3, 2) and cls_vars.shape == (
    3,
    2,
), "Expect one row per class and one column per feature: (3, 2)"
gnb_cls = GaussianNB().fit(X_cls, y_cls)  # the same model from scikit-learn
assert np.allclose(cls_priors, 1 / 3), "Each class is a third of the training flats"
assert np.allclose(
    cls_means, gnb_cls.theta_
), "The means differ from GaussianNB: X[y == c].mean(axis=0)"
assert np.allclose(
    cls_vars, gnb_cls.var_, rtol=1e-4
), "The variances differ from GaussianNB: X[y == c].var(axis=0)"
print(f"{'class':9s} {'mean area':>10s} {'mean price':>11s} {'sd area':>8s} {'sd price':>9s}")
for name, mean, variance in zip(CLASS_NAMES, cls_means, cls_vars, strict=True):
    sd = np.sqrt(variance)
    print(f"{name:9s} {mean[0]:10.1f} {mean[1]:11.0f} {sd[0]:8.1f} {sd[1]:9.1f}")

The log-score of a class adds the logarithm of the prior and the log-densities of the features:

$$\log \text{score}(c) = \log P(c) + \sum_j \left[ -\tfrac{1}{2} \log (2 \pi \sigma_{cj}^2) - \frac{(x_j - \mu_{cj})^2}{2 \sigma_{cj}^2} \right].$$

**Exercise 4.2.** Write `gaussian_nb_log_scores(X, priors, means, variances)`: a matrix with one row per flat and one column per class.

In [ ]:
def gaussian_nb_log_scores(X, priors, means, variances):
    """log P(c) plus the sum over the features of the normal log-density; shape (n, K)."""
    # YOUR CODE HERE: loop over the classes with zip(priors, means, variances, strict=True).
    # A row of means and a row of variances hold one number per feature, so
    # -0.5 * np.log(2 * np.pi * variance) - (X - mean) ** 2 / (2 * variance) is the matrix of
    # log-densities of one class; sum it over axis 1 and add np.log(prior).
    # Return the columns of the classes side by side (np.column_stack)
    return None

In [ ]:
assert gaussian_nb_fit(X_cls, y_cls) is not None, "Finish Exercise 4.1 first"
cls_scores_test = gaussian_nb_log_scores(X_cls_test, cls_priors, cls_means, cls_vars)
assert cls_scores_test is not None, "Fill in gaussian_nb_log_scores above"
assert cls_scores_test.shape == (
    600,
    3,
), "Expect one row per flat and one column per class: (600, 3)"
gnb_cls = GaussianNB().fit(X_cls, y_cls)
assert np.allclose(
    cls_scores_test, gnb_cls.predict_joint_log_proba(X_cls_test), atol=1e-3
), "The log-scores differ from GaussianNB: log prior plus the sum of the log-densities"
cls_pred_test = cls_scores_test.argmax(axis=1)
assert np.array_equal(
    cls_pred_test, gnb_cls.predict(X_cls_test)
), "The predictions differ from GaussianNB"
cls_pred_train = gaussian_nb_log_scores(X_cls, cls_priors, cls_means, cls_vars).argmax(axis=1)
acc_gnb = np.mean(cls_pred_test == y_cls_test)
print(f"accuracy on the 150 training flats: {np.mean(cls_pred_train == y_cls):.3f}")
print(f"accuracy on the 600 new flats:      {acc_gnb:.3f}")
recalls = [np.mean(cls_pred_test[y_cls_test == k] == k) for k in range(3)]
print(
    "recall on the new flats: "
    + ", ".join(f"{name} {value:.3f}" for name, value in zip(CLASS_NAMES, recalls, strict=True))
)

The predictions match `GaussianNB` for all 600 new flats, and the log-scores agree to three decimals. They are not identical because scikit-learn adds a tiny constant to every variance for numerical safety. Gaussian naive Bayes gets 0.883 of the new flats right.

### 4.1 Without the assumption: LDA and QDA

Two classical models keep the normal distributions and drop independence: a class gets a full covariance matrix, with the correlation in it. Quadratic discriminant analysis (QDA) estimates one covariance matrix per class. Linear discriminant analysis (LDA) shares a single matrix between the classes, which makes the boundaries straight lines.

**Exercise 4.3.** Fit `LinearDiscriminantAnalysis()` as `lda` and `QuadraticDiscriminantAnalysis()` as `qda` on the training flats, and store their accuracy on the 600 new flats in `acc_lda` and `acc_qda`.

In [ ]:
# YOUR CODE HERE: fit both models on X_cls, y_cls; model.score(X_cls_test, y_cls_test)
# is the accuracy on the new flats
lda = None
qda = None
acc_lda = None
acc_qda = None

In [ ]:
assert acc_lda is not None and acc_qda is not None, "Fill in lda, qda, acc_lda and acc_qda above"
assert isinstance(lda, LinearDiscriminantAnalysis), "lda is a fitted LinearDiscriminantAnalysis()"
assert isinstance(
    qda, QuadraticDiscriminantAnalysis
), "qda is a fitted QuadraticDiscriminantAnalysis()"
assert np.isclose(
    acc_lda, 0.868, atol=0.002
), "Expect 0.868 for LDA: fit on X_cls, y_cls and score on X_cls_test, y_cls_test"
assert np.isclose(
    acc_qda, 0.870, atol=0.003
), "Expect 0.870 for QDA: fit on X_cls, y_cls and score on X_cls_test, y_cls_test"
gnb_cls = GaussianNB().fit(X_cls, y_cls)
softmax = make_pipeline(  # softmax regression as in week 4
    StandardScaler(), LogisticRegression(C=1e6, tol=1e-10, max_iter=1_000_000)
).fit(X_cls, y_cls)
cls_models = {
    "Gaussian naive Bayes": gnb_cls,
    "LDA, one shared covariance": lda,
    "QDA, a covariance per class": qda,
    "softmax regression (week 4)": softmax,
}
print(f"{'model':29s} {'150 training flats':>19s} {'600 new flats':>14s}")
for name, model in cls_models.items():
    print(
        f"{name:29s} {model.score(X_cls, y_cls):19.3f} {model.score(X_cls_test, y_cls_test):14.3f}"
    )

The next plot shows the regions of the three Gaussian models. The ellipse of a class is the contour two standard deviations away from its mean, under the covariance that the model assumes.

In [ ]:
assert acc_lda is not None and acc_qda is not None, "Finish Exercise 4.3 first"


def distance2(points, mean, cov):
    """Squared distance from the mean in units of the covariance: (x - m)^T cov^-1 (x - m)."""
    diff = points - mean
    return np.einsum("ij,jk,ik->i", diff, np.linalg.inv(cov), diff)


area_grid, ppm_grid = np.meshgrid(np.linspace(15, 125, 221), np.linspace(350, 2350, 201))
grid = np.column_stack([area_grid.ravel(), ppm_grid.ravel()])
class_covs = [np.cov(X_cls[y_cls == k].T, ddof=0) for k in range(3)]  # full covariance per class
assumed = {
    "Gaussian naive Bayes": [np.diag(np.diag(cov)) for cov in class_covs],  # variances only
    "LDA": [np.mean(class_covs, axis=0)] * 3,  # one matrix for all classes
    "QDA": class_covs,
}
panels = [("Gaussian naive Bayes", gnb_cls), ("LDA", lda), ("QDA", qda)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
for ax, (name, model) in zip(axes, panels, strict=True):
    regions = model.predict(grid).reshape(area_grid.shape)
    ax.contourf(
        area_grid, ppm_grid, regions, levels=[-0.5, 0.5, 1.5, 2.5], colors=CLASS_COLOURS, alpha=0.10
    )
    for k in range(3):
        flats = X_cls[y_cls == k]
        ax.scatter(*flats.T, s=12, color=CLASS_COLOURS[k], linewidth=0)
        distance = distance2(grid, flats.mean(axis=0), assumed[name][k]).reshape(area_grid.shape)
        ax.contour(
            area_grid, ppm_grid, distance, levels=[4], colors=[CLASS_COLOURS[k]], linewidths=1.6
        )
    ax.set(xlabel="area, m²", title=f"{name}: {model.score(X_cls_test, y_cls_test):.3f}")
axes[0].set_ylabel("price per m², $")
fig.tight_layout()
plt.show()

On the 600 new flats Gaussian naive Bayes gets 0.883, QDA 0.870, LDA 0.868 and softmax regression 0.872. The ellipses of naive Bayes are aligned with the axes, because the model has no way to tilt them, while those of QDA follow the data. The assumption is plainly wrong here and the classifier is as accurate as the others: the gap between 0.883 and 0.870 is 8 flats out of 600. A label depends only on the side of the boundary, and the boundaries of the three models are close to each other. The page `04_gaussian_naive_bayes.html` shows the bells, the ellipses and the scores of any flat you click.

The accuracy of QDA can differ by one flat between versions of scikit-learn: older versions divide the covariance by $n - 1$ and get 0.868.

## 5. What the assumptions cost

Gaussian naive Bayes makes two assumptions: inside a class every feature is normal, and the features are independent. In section 4 the second one was wrong and the labels survived. The probabilities are the next thing to check. The data is the fake listings of week 5: 2,000 listings to train on and 4,000 to test on, 5% of them fakes. Each has three features: the markup in percent (fakes are too cheap), the number of photos and the age of the seller's account in months.

In [ ]:
FAKE_FEATURES = ["markup", "photos", "age"]


def fake_sample(seed, n, share=0.05):
    """n listings: X = (markup in %, number of photos, account age in months), y = 1 for a fake."""
    rng = np.random.default_rng(seed)
    y = np.zeros(n, dtype=int)
    y[: int(round(share * n))] = 1  # exactly this share of fakes
    rng.shuffle(y)
    markup = np.round(np.where(y == 1, rng.normal(-12, 11, n), rng.normal(4, 9, n)))
    photos = np.where(y == 1, rng.poisson(5, n), rng.poisson(9, n)) + 1
    age = np.round(np.where(y == 1, rng.exponential(8, n), rng.exponential(30, n)) + 1)
    return np.column_stack([markup, photos, age]).astype(float), y


def make_logreg():
    """Logistic regression without a penalty on standardised features, as in week 5."""
    return make_pipeline(
        StandardScaler(), LogisticRegression(C=np.inf, tol=1e-10, max_iter=1_000_000)
    )


X_fake, y_fake = fake_sample(1, 2000)  # the listings we train on
X_fake_test, y_fake_test = fake_sample(5001, 4000)  # the test set

print(
    f"train: {len(y_fake)} listings, {y_fake.sum()} fakes; "
    f"test: {len(y_fake_test)} listings, {y_fake_test.sum()} fakes"
)
print(f"\n{'mean of the feature':20s} " + " ".join(f"{name:>8s}" for name in FAKE_FEATURES))
for label, name in [(1, "fake"), (0, "real")]:
    means = X_fake[y_fake == label].mean(axis=0)
    print(f"{name:20s} " + " ".join(f"{value:8.1f}" for value in means))

**Exercise 5.1.** Fit `GaussianNB()` and `make_logreg()` on the training listings and store their probabilities of "fake" for the test listings in `p_nb` and `p_lr`.

In [ ]:
# YOUR CODE HERE: model.fit(X_fake, y_fake).predict_proba(X_fake_test)[:, 1] for both models
p_nb = None
p_lr = None

In [ ]:
assert p_nb is not None and p_lr is not None, "Fill in p_nb and p_lr above"
assert np.isclose(
    roc_auc_score(y_fake_test, p_nb), 0.966, atol=0.001
), "Expect a ROC-AUC of 0.966 for naive Bayes: fit on X_fake, y_fake, predict X_fake_test"
assert np.isclose(
    roc_auc_score(y_fake_test, p_lr), 0.971, atol=0.001
), "Expect a ROC-AUC of 0.971 for logistic regression: use make_logreg()"
print(
    f"{'model':21s} {'ROC-AUC':>8s} {'log-loss':>9s} {'flagged':>8s} {'precision':>10s} "
    f"{'recall':>7s} {'mean probability':>17s}"
)
for name, p in [("Gaussian naive Bayes", p_nb), ("logistic regression", p_lr)]:
    tp, fp, fn, tn = confusion(y_fake_test, p >= 0.5)
    print(
        f"{name:21s} {roc_auc_score(y_fake_test, p):8.3f} {log_loss(y_fake_test, p):9.3f} "
        f"{tp + fp:8d} {tp / (tp + fp):10.3f} {tp / (tp + fn):7.3f} {p.mean():17.3f}"
    )
print(f"share of fakes among the test listings: {y_fake_test.mean():.3f}")

The ranking is almost as good: a ROC-AUC of 0.966 for naive Bayes against 0.971 for logistic regression. The probabilities are worse, with a log-loss of 0.091 against 0.079. Naive Bayes predicts 0.072 on average where the share of fakes is 0.050 (logistic regression: 0.054), and at the threshold 0.5 it flags 176 listings where logistic regression flags 154.

### 5.1 Calibration

Group the test listings into five bins by predicted probability and compare, in every bin, the mean prediction with the share of listings that are fakes. For a calibrated model the two are equal, as in week 4.

In [ ]:
assert p_nb is not None and p_lr is not None, "Finish Exercise 5.1 first"


def calibration_table(y, p):
    """Five bins of the predicted probability: listings, mean prediction, share of fakes."""
    bins = np.digitize(p, [0.2, 0.4, 0.6, 0.8])  # the number of the bin, 0 to 4, per listing
    return [(int(np.sum(bins == k)), p[bins == k].mean(), y[bins == k].mean()) for k in range(5)]


fig, (ax_cal, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.6))
ax_cal.plot([0, 1], [0, 1], color=INK3, ls="--", lw=1.2, label="calibrated")
for name, p, colour in [
    ("logistic regression", p_lr, ACCENT),
    ("Gaussian naive Bayes", p_nb, ORANGE),
]:
    table = calibration_table(y_fake_test, p)
    print(f"\n{name}\n  bin          listings   mean prediction   share of fakes")
    for k, (n, mean_p, share) in enumerate(table):
        print(f"  {0.2 * k:.1f} to {0.2 * k + 0.2:.1f} {n:10d} {mean_p:17.2f} {share:16.2f}")
    ax_cal.plot(
        [row[1] for row in table], [row[2] for row in table], "o-", color=colour, label=name
    )
    ax_hist.hist(p, bins=np.linspace(0, 1, 21), histtype="step", lw=2, color=colour, label=name)
ax_cal.set(
    xlabel="mean predicted probability in the bin",
    ylabel="share of fakes in the bin",
    title="Calibration on the 4,000 test listings",
)
ax_cal.legend(loc="upper left")
ax_hist.set(
    xlabel="predicted probability",
    ylabel="listings",
    yscale="log",
    title="How often each probability is given",
)
ax_hist.legend(loc="upper center")
fig.tight_layout()
plt.show()

Logistic regression stays near the diagonal. Naive Bayes sits below it in the middle bins: of the 168 listings that it puts between 0.2 and 0.4, a share of 0.17 are fakes, and between 0.4 and 0.6 the share is 0.36. It says "fake" with more confidence than the data supports. On this data the independence assumption holds: inside a class the three features are generated independently of each other. The normal distribution is what fails. The age of an account is exponential and the number of photos is a count, so the bell curves fit them badly, and an error in a likelihood goes straight into the probability.

### 5.2 Counting the same evidence twice

To see what dependence does, make two features as dependent as they can be: add a copy of the markup column to the data. It brings no new information. Logistic regression can split the weight between the twins, as ridge did in week 3. Naive Bayes multiplies the likelihood of the markup into the score twice.

**Exercise 5.2.** For 1, 2, 4 and 8 copies of the markup column, fit `GaussianNB()` and `make_logreg()` on the widened training data and store their test probabilities in the dictionaries `p_nb_copies` and `p_lr_copies` under the number of copies. `repeat_markup` widens a matrix.

In [ ]:
def repeat_markup(X, copies):
    """X with the markup (column 0) present `copies` times; the extra copies go to the right."""
    return np.column_stack([X] + [X[:, [0]]] * (copies - 1))


p_nb_copies, p_lr_copies = {}, {}
for copies in [1, 2, 4, 8]:
    X_wide, X_wide_test = repeat_markup(X_fake, copies), repeat_markup(X_fake_test, copies)
    # YOUR CODE HERE: as in Exercise 5.1, with X_wide and X_wide_test in place of X_fake and
    # X_fake_test: p_nb_copies[copies] = ..., p_lr_copies[copies] = ...

In [ ]:
assert (
    len(p_nb_copies) == 4 and len(p_lr_copies) == 4
), "Fill in the loop above: one array of probabilities per number of copies in each dictionary"
flagged = [int(np.sum(p_nb_copies[copies] >= 0.5)) for copies in [1, 2, 4, 8]]
assert flagged == [
    176,
    244,
    326,
    449,
], "Expect naive Bayes to flag 176, 244, 326 and 449 listings: fit on X_wide, predict X_wide_test"
share_loss = log_loss(y_fake_test, np.full(len(y_fake_test), y_fake.mean()))
print(
    "copies   flagged   precision   recall   log-loss   ROC-AUC   log-loss of logistic regression"
)
for copies in [1, 2, 4, 8]:
    p = p_nb_copies[copies]
    tp, fp, fn, tn = confusion(y_fake_test, p >= 0.5)
    print(
        f"{copies:6d} {tp + fp:9d} {tp / (tp + fp):11.3f} {tp / (tp + fn):8.3f} "
        f"{log_loss(y_fake_test, p):10.3f} {roc_auc_score(y_fake_test, p):9.3f} "
        f"{log_loss(y_fake_test, p_lr_copies[copies]):33.3f}"
    )
print(f"log-loss of answering {y_fake.mean():.2f} for every listing: {share_loss:.3f}")

fig, (ax_loss, ax_cal) = plt.subplots(1, 2, figsize=(13, 4.6))
ax_loss.plot(
    [1, 2, 4, 8],
    [log_loss(y_fake_test, p_nb_copies[c]) for c in [1, 2, 4, 8]],
    "o-",
    color=ORANGE,
    label="Gaussian naive Bayes",
)
ax_loss.plot(
    [1, 2, 4, 8],
    [log_loss(y_fake_test, p_lr_copies[c]) for c in [1, 2, 4, 8]],
    "o-",
    color=ACCENT,
    label="logistic regression",
)
ax_loss.axhline(share_loss, color=INK3, ls="--", lw=1.2, label="always 0.05")
ax_loss.set(
    xlabel="copies of the markup column",
    ylabel="log-loss on the test listings",
    xticks=[1, 2, 4, 8],
    ylim=(0, 0.65),
    title="Log-loss",
)
ax_loss.legend(loc="upper left")
ax_cal.plot([0, 1], [0, 1], color=INK3, ls="--", lw=1.2, label="calibrated")
for copies, colour in [(1, ORANGE), (8, PURPLE)]:
    bins = np.digitize(p_nb_copies[copies], [0.2, 0.4, 0.6, 0.8])
    mean_p = [p_nb_copies[copies][bins == k].mean() for k in range(5)]
    share = [y_fake_test[bins == k].mean() for k in range(5)]
    ax_cal.plot(
        mean_p,
        share,
        "o-",
        color=colour,
        label=f"naive Bayes, {copies} " f"{'copy' if copies == 1 else 'copies'}",
    )
ax_cal.set(
    xlabel="mean predicted probability in the bin",
    ylabel="share of fakes in the bin",
    title="Calibration of naive Bayes",
)
ax_cal.legend(loc="upper left")
fig.tight_layout()
plt.show()

With 8 copies naive Bayes flags 449 listings where it flagged 176, its precision falls from 0.733 to 0.332 and its log-loss grows from 0.091 to 0.582. Answering 0.05 for every listing would score 0.199, so these probabilities are worse than no model. The ranking suffers less: the ROC-AUC goes from 0.966 to 0.933. Logistic regression stays at 0.079 in every row. Real features are rarely exact copies, but correlated features repeat a part of each other, and naive Bayes counts that part more than once. The page `05_double_counting.html` has the number of copies as a control.

## 6. Three classifiers, one dataset

Which classifier should the site use: kNN from week 1, logistic regression or naive Bayes? The protocol is the one of weeks 3 and 5. The metric is chosen in advance: average precision, because fakes are rare and the moderators read a ranked list. The models are compared by stratified 5-fold cross-validation on the 2,000 training listings, with the same folds for every model. The test set is opened once, at the end.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)  # the same folds for every model
K_GRID = [1, 3, 5, 9, 15, 25, 41]
SCORING = ["roc_auc", "average_precision", "f1", "neg_log_loss"]


def make_knn(k):
    """kNN on standardised features: distances need a common scale (week 1)."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))


for number, (train_idx, val_idx) in enumerate(folds.split(X_fake, y_fake), start=1):
    print(
        f"fold {number}: train on {len(train_idx)} listings, validate on {len(val_idx)}, "
        f"{y_fake[val_idx].sum()} of them fakes"
    )

kNN has a hyperparameter, so it goes first.

**Exercise 6.1.** For every `k` in `K_GRID`, compute the cross-validated average precision of `make_knn(k)` with `cross_val_score(..., cv=folds, scoring="average_precision")` and append the mean over the folds to `knn_cv`.

In [ ]:
knn_cv = []
# YOUR CODE HERE: for every k in K_GRID call cross_val_score(make_knn(k), X_fake, y_fake, ...)
# and append the mean of the five scores to knn_cv

In [ ]:
assert len(knn_cv) == len(K_GRID), "Fill in the loop above: one number per k in knn_cv"
first = cross_val_score(make_knn(1), X_fake, y_fake, cv=folds, scoring="average_precision")
assert np.isclose(knn_cv[0], first.mean()), (
    "The first number is the mean of the five fold scores of make_knn(1): cv=folds, "
    "scoring='average_precision'"
)
assert np.allclose(
    knn_cv, [0.241, 0.446, 0.535, 0.581, 0.597, 0.639, 0.661], atol=0.002
), "Expect 0.241 at k = 1 and 0.661 at k = 41: one mean over the folds for every k"
best_k = K_GRID[int(np.argmax(knn_cv))]
print("  k   cross-validated average precision")
for k, value in zip(K_GRID, knn_cv, strict=True):
    print(f"{k:3d} {value:10.3f}")
print(f"best k: {best_k}")

models = {
    f"kNN, k = {best_k}": make_knn(best_k),
    "logistic regression": make_logreg(),
    "Gaussian naive Bayes": GaussianNB(),
}

# cross_validate is cross_val_score for several metrics at once
demo = cross_validate(GaussianNB(), X_fake, y_fake, cv=folds, scoring=SCORING)
print(
    "\nkeys of a cross_validate result:", ", ".join(key for key in demo if key.startswith("test"))
)
print("test_average_precision of Gaussian naive Bayes:", demo["test_average_precision"].round(3))

The average precision grows along the whole grid: 0.241 at $k = 1$, 0.597 at $k = 15$ and 0.661 at $k = 41$, so we take $k = 41$. With 5% of fakes a small neighbourhood holds too few of them to give a useful score, and at $k = 1$ the score is only ever 0 or 1. The best value sits on the edge of the grid, so a larger $k$ may do better still (see "Try at home").

`cross_validate` returns a dictionary with one array of five fold scores per metric. scikit-learn maximises scores, so the log-loss comes back with a minus sign, as the MSE did in week 3.

**Exercise 6.2.** Run `cross_validate` for every model in `models` with `cv=folds` and `scoring=SCORING`, and store the result of each call in `cv_results` under the name of the model.

In [ ]:
cv_results = {}
# YOUR CODE HERE: for name, model in models.items(): cv_results[name] = cross_validate(...)

In [ ]:
assert len(cv_results) == 3, "Fill in the loop above: one cross_validate result per model"
cv_ap = {name: result["test_average_precision"] for name, result in cv_results.items()}
assert np.allclose(
    [scores.mean() for scores in cv_ap.values()], [0.661, 0.712, 0.688], atol=0.002
), "Expect a mean average precision of 0.661, 0.712 and 0.688: cv=folds, scoring=SCORING"
print(
    f"{'mean over the folds':21s} {'ROC-AUC':>8s} {'average precision':>18s} {'F1':>6s} {'log-loss':>9s}"
)
for name, result in cv_results.items():
    print(
        f"{name:21s} {result['test_roc_auc'].mean():8.3f} "
        f"{result['test_average_precision'].mean():18.3f} {result['test_f1'].mean():6.3f} "
        f"{-result['test_neg_log_loss'].mean():9.3f}"
    )
print(
    f"\n{'average precision':21s} " + " ".join(f"fold {k}" for k in range(1, 6)) + "   spread (sd)"
)
for name, scores in cv_ap.items():
    print(f"{name:21s} " + " ".join(f"{value:6.3f}" for value in scores) + f" {scores.std():13.3f}")

fig, ax = plt.subplots(figsize=(8.5, 4.5))
for (name, scores), colour in zip(cv_ap.items(), [GREEN, ACCENT, ORANGE], strict=True):
    ax.plot(range(1, 6), scores, "o-", color=colour, label=f"{name}: mean {scores.mean():.3f}")
ax.set(
    xlabel="fold",
    ylabel="average precision",
    xticks=range(1, 6),
    ylim=(0.4, 0.95),
    title="The same five folds for the three models",
)
ax.legend(loc="upper left", ncols=1)
plt.show()

By mean average precision the order is logistic regression 0.712, naive Bayes 0.688, kNN 0.661. Now look at the folds. In fold 5 every model falls to about 0.5, and in fold 3 every model is above 0.77. The spread between the folds, a standard deviation of about 0.11 for each model, is larger than any gap between the models. A fold has 20 fakes, and average precision on 20 positives is a noisy number. The comparison still means something because the folds are shared: logistic regression beats kNN in all five folds and naive Bayes in four.

Every choice is made, so the test set can be opened.

In [ ]:
assert len(cv_results) == 3, "Finish Exercise 6.2 first"
test_ap = {}
print(
    f"{'4,000 test listings':21s} {'ROC-AUC':>8s} {'average precision':>18s} {'precision':>10s} "
    f"{'recall':>7s} {'log-loss':>9s}"
)
for name, model in models.items():
    p = model.fit(X_fake, y_fake).predict_proba(X_fake_test)[:, 1]  # refit on all 2,000 listings
    tp, fp, fn, tn = confusion(y_fake_test, p >= 0.5)
    test_ap[name] = average_precision_score(y_fake_test, p)
    print(
        f"{name:21s} {roc_auc_score(y_fake_test, p):8.3f} {test_ap[name]:18.3f} "
        f"{tp / (tp + fp):10.3f} {tp / (tp + fn):7.3f} {log_loss(y_fake_test, p):9.3f}"
    )

On the 4,000 test listings the average precision is 0.752 for logistic regression, 0.739 for naive Bayes and 0.723 for kNN, the order that cross-validation gave. At the threshold 0.5 the models behave differently. kNN is the most cautious, with a precision of 0.849 and a recall of 0.450. Naive Bayes flags the most listings and gets 0.733 and 0.645. Its log-loss of 0.091 is better than the 0.114 of kNN and worse than the 0.079 of logistic regression.

### 6.1 Practice: real data

The last dataset is real: 569 breast tumours with 30 measurements each, bundled with scikit-learn. The positive class here is "malignant". A quarter of the data is held out as the test set.

In [ ]:
cancer = load_breast_cancer()  # bundled with scikit-learn, no download
malignant = 1 - cancer.target  # scikit-learn codes malignant as 0; here it is the positive class
X_tum, X_tum_test, y_tum, y_tum_test = train_test_split(
    cancer.data, malignant, test_size=0.25, stratify=malignant, random_state=SEED
)
print(f"{len(malignant)} tumours, {cancer.data.shape[1]} features, {malignant.sum()} malignant")
print(
    f"train: {len(y_tum)} tumours, {y_tum.sum()} malignant; "
    f"test: {len(y_tum_test)} tumours, {y_tum_test.sum()} malignant"
)

**Exercise 6.3.** Repeat the protocol on the training tumours. Choose `tumour_k` from `K_GRID` by cross-validated average precision of `make_knn(k)`, as in Exercise 6.1. Then run `cross_validate` with `cv=folds` and `scoring=SCORING` for three models and store the results in the dictionary `tumour_cv` under these names: `"kNN"` for `make_knn(tumour_k)`, `"logistic regression"` for `make_pipeline(StandardScaler(), LogisticRegression())` and `"Gaussian naive Bayes"` for `GaussianNB()`.

In [ ]:
# YOUR CODE HERE: the loop of Exercise 6.1 on X_tum, y_tum gives tumour_k; then the loop of
# Exercise 6.2 over the three models fills tumour_cv
tumour_k = None
tumour_cv = {}

In [ ]:
names = ["kNN", "logistic regression", "Gaussian naive Bayes"]
assert tumour_k is not None and len(tumour_cv) == 3, "Fill in tumour_k and tumour_cv above"
assert sorted(tumour_cv) == sorted(
    names
), 'The keys of tumour_cv are "kNN", "logistic regression" and "Gaussian naive Bayes"'
assert (
    tumour_k == 25
), "Expect k = 25: the largest mean of cross_val_score over K_GRID on X_tum, y_tum"
tumour_ap = [tumour_cv[name]["test_average_precision"].mean() for name in names]
assert np.allclose(
    tumour_ap, [0.988, 0.991, 0.981], atol=0.003
), "Expect a mean average precision of 0.988, 0.991 and 0.981: cv=folds, scoring=SCORING"
print(f"k for kNN: {tumour_k}\n")
print(
    f"{'mean over the folds':21s} {'ROC-AUC':>8s} {'average precision':>18s} {'F1':>6s} {'log-loss':>9s}"
)
for name in names:
    result = tumour_cv[name]
    print(
        f"{name:21s} {result['test_roc_auc'].mean():8.3f} "
        f"{result['test_average_precision'].mean():18.3f} {result['test_f1'].mean():6.3f} "
        f"{-result['test_neg_log_loss'].mean():9.3f}"
    )

final_models = {
    "kNN": make_knn(tumour_k),
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression()),
    "Gaussian naive Bayes": GaussianNB(),
}
print(
    f"\n{'143 test tumours':21s} {'ROC-AUC':>8s} {'average precision':>18s} {'mistakes':>9s} "
    f"{'log-loss':>9s} {'below 0.01 or above 0.99':>25s}"
)
for name, model in final_models.items():
    p = model.fit(X_tum, y_tum).predict_proba(X_tum_test)[:, 1]
    extreme = np.mean((p < 0.01) | (p > 0.99))
    print(
        f"{name:21s} {roc_auc_score(y_tum_test, p):8.3f} "
        f"{average_precision_score(y_tum_test, p):18.3f} {np.sum((p >= 0.5) != y_tum_test):9d} "
        f"{log_loss(y_tum_test, p):9.3f} {extreme:25.3f}"
    )

Logistic regression uses its default penalty here, since 30 features on 426 tumours are close to separable. All three models rank the tumours well: the cross-validated average precision is 0.988 for kNN, 0.991 for logistic regression and 0.981 for naive Bayes, and on the 143 test tumours it is 0.996, 0.994 and 0.992. The log-loss tells them apart. In cross-validation it is 0.092 for logistic regression and 0.580 for naive Bayes. Many of the 30 features are near twins, for example the radius, the perimeter and the area of a nucleus, so naive Bayes counts the same evidence several times, as in section 5.2. On the test tumours 96% of its probabilities are below 0.01 or above 0.99. Its ranking can be used, and its probabilities should not be read as probabilities.

## 7. What to remember

- Bayes' rule: the posterior is the prior times the likelihood, divided by the evidence. The same evidence gives another answer when the prior changes.
- Naive Bayes assumes that the features are independent given the class. Then $P(x \mid c)$ is a product of one-feature terms, and training is counting words or taking means and variances. Compute in logarithms.
- Without smoothing, one unseen word sets the score of a class to zero. Laplace smoothing adds $\alpha$ to every count.
- With two classes the log-odds of multinomial naive Bayes are linear in the word counts, with the weights $\log(\theta_{\text{fake}, j} / \theta_{\text{real}, j})$.
- Wrong assumptions often leave the labels and the ranking almost intact and damage the probabilities. A likelihood of the wrong shape distorts them, and correlated features are counted more than once. Check the calibration and the log-loss before you use the probabilities.
- Compare classifiers by cross-validation on shared stratified folds with a metric chosen in advance, look at the spread between the folds, and open the test set once.

## Try at home

1. In section 2, classify the listing "metro balcony prepayment" with $\alpha = 0$ and with $\alpha = 1$. Which class wins in each case, and why?
2. In section 3, train on the first 20 listings only (`X_text[:20]`, `y_text[:20]`) and compare $\alpha = 0.01$, 1 and 10 on the test listings. Does $\alpha$ matter more than it did with 200 listings?
3. In Exercise 6.1, extend the grid with $k = 61$, 81, 121, 201 and 301. Where is the maximum, and is the gain larger than the spread between the folds?
4. `CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5)` from `sklearn.calibration` learns a correction of the probabilities by cross-validation. Fit it on the data with 8 copies of the markup. What happens to the log-loss, and what happens to the ROC-AUC?